# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [5]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Gemini API Key exists and begins {google_api_key[:8]}")
else:
    print("Gemini API Key not set")

Gemini API Key exists and begins AQ.Ab8RN


In [6]:
# Initialize
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
MODEL = 'gemini-3.5-flash-lite'

In [7]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [8]:
def chat(message, history):
    return "bananas"

In [9]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!

In [14]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(history)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I help you today?'}]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I help you today?'}, {'role': 'user', 'content': 'I am testing gradio'}, {'role': 'assistant', 'content': "That's awesome! Gradio is a fantastic tool for building quick machine learning demos and web UIs with Python. \n\nHow is the testing going? Are you trying out a simple text input, an image classifier, or something else? Let me know if you need any code snippets or help with your app!"}]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I help you today?'}, {'role': 'user', 'content': 'I am testing gradio'}, {'role': 'assistant', 'content': "That's awesome! Gradio is a fantastic tool for building quick machine learning demos and web UIs with Python. \n\nHow is the testing going? Are you trying out a simple text input, an image classifier, or something else?

In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(history)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [17]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I help you today?'}]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I help you today?'}, {'role': 'user', 'content': 'I am feeling overwhelmed'}, {'role': 'assistant', 'content': "I'm really sorry to hear that you're feeling this way. Take a deep breath. It's completely okay to feel overwhelmed sometimes.\n\nWould it help to talk about what's on your mind, or would you prefer to break things down into smaller, manageable steps? I'm here to listen or help however I can."}]


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [18]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [19]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': "Hello! Welcome to our store. How can I help you find what you're looking for today? \n\nIf you're not quite sure what you need, I always love to suggest our fantastic selection of hats—they're currently a whopping 60% off!"}]
[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': "Hello! Welcome to our store. How can I help you find what you're looking for today? \n\nIf you're not quite sure what you need, I always love to suggest our fantastic selection of hats—they're currently a whopping 60% off!"}, {'role': 'user', 'content': 'No discounts on other items only hats?'}, {'role': 'assistant', 'content': 'Oh, we definitely have discounts on other items too! Most of our store is actually 50% off right now as part of our big sales event. \n\nHowever, our hats are getting an extra special markdown at 60% off, which makes them an absolute steal! Are you looking for anything specific today, like a cozy wi

In [20]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': "Hello! Welcome to our store. How can I help you find what you're looking for today? \n\nIf you're not quite sure what you need, I always love to suggest our fantastic selection of hats—they're currently a whopping 60% off!"}, {'role': 'user', 'content': 'No discounts on other items only hats?'}, {'role': 'assistant', 'content': 'Oh, we definitely have discounts on other items too! Most of our store is actually 50% off right now as part of our big sales event. \n\nHowever, our hats are getting an extra special markdown at 60% off, which makes them an absolute steal! Are you looking for anything specific today, like a cozy winter hat, a stylish cap, or maybe an outfit to go with a new accessory?'}, {'role': 'user', 'content': 'Why are you only encouraging hats? Thats not what I am looking for'}, {'role': 'assistant', 'content': "I am so sorry about that! I didn't mean to laser in too much on the hats—I just get a bit e

In [21]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [23]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [24]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [25]:
gr.close_all()

Closing server running on port: 7862
Closing server running on port: 7866
Closing server running on port: 7864
Closing server running on port: 7861
Closing server running on port: 7863
Closing server running on port: 7865
Closing server running on port: 7860


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>